# Raw RSSI extractor (AEP_OK)
Minimal, readable pipeline that mirrors RTLS_SF_1tower.ipynb core logic.
Output: long dataframe with one row per (timestamp, Tx, rx) reading and x,y,z labels.

## CODE: Imports and settings

In [15]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

RAW_ROOT = Path('../raw_data')
INVALID_RSSI_DBM = -100.0

## CODE: Helpers

In [16]:
def parse_datestamp(value: str) -> pd.Timestamp:
    return pd.to_datetime(value, format='%y/%m/%d %H:%M:%S.%f')


def parse_file_metadata(file_path: Path) -> dict:
    parts = file_path.parts
    idx = parts.index('raw_data')
    conf = parts[idx + 1]
    pgm = parts[idx + 2]
    run_id = parts[idx + 3]
    tokens = {}
    for token in file_path.stem.split('__'):
        if '=' in token:
            key, val = token.split('=', 1)
            tokens[key] = val
    return {
        'conf': conf,
        'pgm': pgm,
        'run_id': run_id,
        'x': float(tokens['x']),
        'y': float(tokens['y']),
        'z0': float(tokens['z0']),
        'txPower_dBm_file': float(tokens['P']),
        'file_name': file_path.name,
    }


def parse_txt_file(file_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    meta = parse_file_metadata(file_path)
    tags_rows = []
    round_rows = []
    run_ts = None
    with file_path.open('rt') as handle:
        for line in handle:
            if not line.startswith('data: '):
                continue
            data_str = line.split('data: ', 1)[1]
            try:
                data_json = json.loads(data_str)
            except json.JSONDecodeError:
                continue  # skip truncated / malformed lines
            if run_ts is None:
                run_ts = parse_datestamp(data_json['datestamp']).round('1s')
            data_type = data_json.get('type')
            if data_type == 'TagReadData':
                tags_rows.append(data_json)
            elif data_type == 'RoundStart':
                round_rows.append(data_json)
    tags_df = pd.DataFrame(tags_rows)
    round_df = pd.DataFrame(round_rows)
    tags_df['run'] = run_ts
    round_df['run'] = run_ts
    tags_df['conf'] = meta['conf']
    round_df['conf'] = meta['conf']
    run_info = {
        'conf': meta['conf'],
        'run': run_ts,
        'x': meta['x'],
        'y': meta['y'],
        'z0': meta['z0'],
    }
    return tags_df, round_df, run_info


def load_raw_data(raw_root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    tag_frames = []
    round_frames = []
    runs = []
    for file_path in sorted(raw_root.rglob('*.txt')):
        if 'pgm=AEP_OK' not in file_path.parts:
            continue
        tags_df, round_df, run_info = parse_txt_file(file_path)
        tag_frames.append(tags_df)
        round_frames.append(round_df)
        runs.append(run_info)
    tags = pd.concat(tag_frames, ignore_index=True)
    roundstart = pd.concat(round_frames, ignore_index=True)
    runs_df = pd.DataFrame(runs).drop_duplicates()
    return tags, roundstart, runs_df


def expand_roundstart(roundstart: pd.DataFrame) -> pd.DataFrame:
    def rx_port(cfg: dict, idx: int) -> str:
        return f"{cfg[f'antennaPort{idx}']}__{cfg[f'expanderPort{idx}']}"

    roundstart = roundstart.copy()
    roundstart['Tx'] = roundstart[['txAntennaPort', 'txExpanderPort']].apply(
        lambda x: '__'.join(x), axis=1
    )
    roundstart['rx1_ant'] = roundstart['rxAntennaConfig'].apply(lambda x: rx_port(x, 1))
    roundstart['rx2_ant'] = roundstart['rxAntennaConfig'].apply(lambda x: rx_port(x, 2))
    roundstart['rx1_rx2_ant'] = roundstart['rx1_ant'] + '|' + roundstart['rx2_ant']
    keep_cols = [
        'conf',
        'run',
        'round',
        'Tx',
        'rx1_ant',
        'rx2_ant',
        'rx1_rx2_ant',
        'txPower_dBm',
        'freq_MHz',
    ]
    return roundstart[keep_cols]


def extract_epc(value: str) -> str:
    return value[6:30]


def split_rssi_pair(value) -> tuple[float, float]:
    return float(value[0]), float(value[1])


def load_actuals(raw_root: Path) -> pd.DataFrame:
    frames = []
    for conf_dir in raw_root.iterdir():
        if not conf_dir.is_dir():
            continue
        actuals_dir = conf_dir / 'Actuals'
        if not actuals_dir.exists():
            continue
        for csv_path in actuals_dir.glob('*.csv'):
            df = pd.read_csv(csv_path, sep=';', decimal=',', dtype=str)
            df = df[['EPC', 'z_rel']].copy()
            df['conf'] = conf_dir.name
            frames.append(df)
    actuals = pd.concat(frames, ignore_index=True)
    actuals['z_rel'] = actuals['z_rel'].astype(str).str.replace(',', '.').astype(float)
    return actuals

## CODE: Build long dataset

In [17]:
tags_raw, roundstart_raw, runs_df = load_raw_data(RAW_ROOT)

roundstart = expand_roundstart(roundstart_raw)

tags = tags_raw.merge(
    roundstart,
    on=['conf', 'run', 'round'],
    how='left',
)

tags['EPC'] = tags['data'].apply(extract_epc)
tags['datestamp'] = pd.to_datetime(
    tags['datestamp'],
    format='%y/%m/%d %H:%M:%S.%f',
)

tags['rssi1_dBm'] = tags['rssi'].str[0].astype(float)
tags['rssi2_dBm'] = tags['rssi'].str[1].astype(float)

tags_long = pd.melt(
    tags,
    id_vars=[
        'conf',
        'run',
        'datestamp',
        'EPC',
        'Tx',
        'rx1_ant',
        'rx2_ant',
        'rx1_rx2_ant',
        'txPower_dBm',
    ],
    value_vars=['rssi1_dBm', 'rssi2_dBm'],
    var_name='rssi_slot',
    value_name='rssi_dBm',
)

tags_long.loc[tags_long['rssi_slot'] == 'rssi1_dBm', 'rx'] = tags_long['rx1_ant']
tags_long.loc[tags_long['rssi_slot'] == 'rssi2_dBm', 'rx'] = tags_long['rx2_ant']

tags_long = tags_long[tags_long['rssi_dBm'] > INVALID_RSSI_DBM].copy()

tags_long['rssi_linear'] = 1e6 * 10 ** (tags_long['rssi_dBm'] / 10)

tags_long = tags_long.drop(columns=['rx1_ant', 'rx2_ant', 'rssi_slot'])

actuals = load_actuals(RAW_ROOT)

runs_small = runs_df[['conf', 'run', 'x', 'y', 'z0']].drop_duplicates()

actuals_runs = actuals.merge(runs_small, on=['conf'])
actuals_runs['z'] = actuals_runs['z0'] + actuals_runs['z_rel']
actuals_runs = actuals_runs[['conf', 'run', 'EPC', 'x', 'y', 'z']]

dataset = tags_long.merge(
    actuals_runs,
    on=['conf', 'run', 'EPC'],
    how='inner',
)

dataset = dataset[[
    'conf',
    'run',
    'datestamp',
    'EPC',
    'Tx',
    'rx',
    'txPower_dBm',
    'rssi_dBm',
    'rssi_linear',
    'x',
    'y',
    'z',
]].sort_values(['run', 'datestamp', 'Tx', 'rx']).reset_index(drop=True)

dataset.head()

,conf,run,datestamp,EPC,Tx,rx,txPower_dBm,rssi_dBm,rssi_linear,x,y,z
0,ConfID=2026-02-03__site=HQ,2026-02-04 21:23:51,2026-02-04 21:23:50.647,BA0507000000000000000000,041A9F60__NONE,PORT_3__NONE,31.5,-77.97,0.015959,1.82,2.51,1.34
1,ConfID=2026-02-03__site=HQ,2026-02-04 21:23:51,2026-02-04 21:23:50.647,BA0604000000000000000000,041A9F60__NONE,PORT_3__NONE,31.5,-76.42,0.022803,1.82,2.51,1.67
2,ConfID=2026-02-03__site=HQ,2026-02-04 21:23:51,2026-02-04 21:23:50.647,BA0507000000000000000000,041A9F60__NONE,PORT_4__NONE,31.5,-83.32,0.004656,1.82,2.51,1.34
3,ConfID=2026-02-03__site=HQ,2026-02-04 21:23:51,2026-02-04 21:23:50.647,BA0604000000000000000000,041A9F60__NONE,PORT_4__NONE,31.5,-75.39,0.028907,1.82,2.51,1.67
4,ConfID=2026-02-03__site=HQ,2026-02-04 21:23:51,2026-02-04 21:23:50.648,BA0508000000000000000000,041A9F60__NONE,PORT_3__NONE,31.5,-76.44,0.022699,1.82,2.51,1.34


In [19]:
dataset.shape

(730081, 12)

In [22]:
dataset['datestamp'].max()

Timestamp('2026-02-17 22:03:25.576000')

In [23]:
dataset['datestamp'].min()

Timestamp('2026-02-04 21:23:50.647000')